# Hotel Booking Cancellation Predictor

A deep learning binary classifier predicting whether a hotel booking will be cancelled.

**Stack:** Python · TensorFlow/Keras · Scikit-learn · Pandas  
**Dataset:** Hotel Bookings (~119,000 records)  
**Model:** Deep Neural Network with BatchNorm + Dropout  

## 1. Imports & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from tensorflow import keras
from tensorflow.keras import layers

plt.style.use('seaborn-v0_8-whitegrid')
plt.rc('figure', autolayout=True)
plt.rc('axes', labelweight='bold', labelsize='large', titleweight='bold', titlesize=18, titlepad=10)


## 2. Load & Explore Data

In [ ]:
hotel = pd.read_csv('hotel_bookings.csv')

X = hotel.copy()
y = X.pop('is_canceled')

print(f'Dataset shape: {hotel.shape}')
print(f'Cancellation rate: {y.mean():.1%}')
hotel.head()


## 3. Feature Engineering & Preprocessing

- **Numerical**: impute → standardize
- **Categorical**: impute → one-hot encode
- Month names mapped to integers
- `random_state=42` ensures reproducibility

In [ ]:
# Map month names to integers
X['arrival_date_month'] = X['arrival_date_month'].map(
    {'January':1, 'February':2, 'March':3, 'April':4, 'May':5, 'June':6,
     'July':7, 'August':8, 'September':9, 'October':10, 'November':11, 'December':12}
)

features_num = [
    'lead_time', 'arrival_date_week_number', 'arrival_date_day_of_month',
    'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children',
    'babies', 'is_repeated_guest', 'previous_cancellations',
    'previous_bookings_not_canceled', 'required_car_parking_spaces',
    'total_of_special_requests', 'adr',
]
features_cat = [
    'hotel', 'arrival_date_month', 'meal', 'market_segment',
    'distribution_channel', 'reserved_room_type', 'deposit_type', 'customer_type',
]

preprocessor = make_column_transformer(
    (make_pipeline(SimpleImputer(strategy='constant'), StandardScaler()), features_num),
    (make_pipeline(SimpleImputer(strategy='constant', fill_value='NA'), OneHotEncoder(handle_unknown='ignore')), features_cat),
)

# Stratified split with fixed random state for reproducibility
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, stratify=y, train_size=0.75, random_state=42
)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)

input_shape = [X_train.shape[1]]
print(f'Input features after encoding: {input_shape[0]}')


## 4. Model Architecture

- **BatchNorm** at input and after each Dense layer
- **Dropout(0.3)** prevents overfitting
- **Sigmoid** output for binary classification probability

In [ ]:
model = keras.Sequential([
    layers.BatchNormalization(input_shape=input_shape),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['binary_accuracy'],
)

model.summary()


## 5. Training with Early Stopping

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=5,
    min_delta=0.001,
    restore_best_weights=True,
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    batch_size=512,
    epochs=200,
    callbacks=[early_stopping],
)


## 6. Results

In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
history_df[['loss', 'val_loss']].plot(ax=axes[0], title='Cross-Entropy Loss')
history_df[['binary_accuracy', 'val_binary_accuracy']].plot(ax=axes[1], title='Accuracy')
plt.tight_layout()
plt.show()

print(f'Best Validation Accuracy: {history_df["val_binary_accuracy"].max():.4f}')
print(f'Best Validation Loss:     {history_df["val_loss"].min():.4f}')


## 7. Save & Download Model

In [ ]:
# Save model and preprocessor
model.save('model_final.keras')
joblib.dump(preprocessor, 'preprocessor_final.pkl')
print('Saved successfully!')
print(f'Model input shape: {input_shape}')


In [ ]:
# Download files to your computer
from google.colab import files
files.download('model_final.keras')
files.download('preprocessor_final.pkl')
